In [1]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-

'''
Created on 2024-07-03
Last modified on 2024-07-03
@author: Juan Enrique López
@description: Jupyter Notebook creado para obtener las relaciones entre técnicas, tácticas, grupos, software, 
@reference documentantion: https://github.com/mitre-attack/attack-stix-data/blob/master/USAGE.md
'''

'\nCreated on 2024-07-03\nLast modified on 2024-07-03\n@author: Juan Enrique López\n@description: Jupyter Notebook creado para obtener las relaciones entre técnicas, tácticas, grupos, software, \n@reference documentantion: https://github.com/mitre-attack/attack-stix-data/blob/master/USAGE.md\n'

#### **Requerimientos**

In [2]:
from stix2 import Filter, MemoryStore
import stix2
import requests

import os
import pandas as pd

import datetime
import csv

#### **Funciones**

In [3]:
def create_output_folder(path):
    '''
    Función encargada para crear el directorio facilitado en caso de no existir previamente.
    '''
    if not os.path.exists(path):
        os.makedirs(path)

In [4]:
def get_list_of_files_sub(dir_name):
    '''
    Función encargada de retornar una lista de archivos ubicados en la ruta facilitada así como en los subdirectorios disponibles.
    '''
    listOfFile = os.listdir(dir_name)
    allFiles = list()
    for entry in listOfFile:
        fullPath = os.path.join(dir_name, entry)
        if os.path.isdir(fullPath):
            allFiles = allFiles + get_list_of_files_sub(fullPath)
        else:
            allFiles.append(fullPath)
    return allFiles

In [5]:
def get_unique_ttps_generated(paths):
    '''
    Función encargada de retornar una lista de id de TTP's únicas generadas. Esta función recibirá una ruta generada previamente en la estructura de carpetas de guardado. 
    '''
    sub_folders = set()
    for path in paths:
        dir, file = os.path.split(path)
        last_sub_folder = os.path.basename(dir).strip()
        sub_folders.add(last_sub_folder)
    return list(sub_folders)

In [6]:
def compare_lists(list1, list2):
    '''
    Función para comparar el contenido de dos listas.
    '''
    set1 = set(list1)
    set2 = set(list2)
    # Items en comun
    common_elements = set1.intersection(set2)
    # Items unicamente en list1
    unique_in_list1 = set1.difference(set2)
    # Items unicamente en list2
    unique_in_list2 = set2.difference(set1)
    # Items en común
    num_common_elements = len(common_elements)
    # Items no coincidentes
    num_non_common_elements = len(unique_in_list1) + len(unique_in_list2)
    return num_common_elements, list(common_elements), list(unique_in_list1), list(unique_in_list2), num_non_common_elements

In [7]:
def unique_list(series):
    '''
    Función para devolver sólo items únicos al agregar
    '''
    return list(set(series))

In [8]:
def get_data_from_branch(matrix):
    '''
    Función encargada de peticionar a la url de GitHub donde está publicada la última versión MITRE de la información en formato stix2. Retorna el objeto que contiene toda la información de MITRE.
    '''
    url = f"https://raw.githubusercontent.com/mitre/cti/master/{matrix}-attack/{matrix}-attack.json"
    
    try:
        response = requests.get(url)
        response.raise_for_status()  # Verificar si la solicitud fue exitosa
        stix_json = response.json()
        
        if "objects" in stix_json:
            return MemoryStore(stix_data=stix_json["objects"])
        else:
            raise ValueError("JSON no contiene la clave 'objects'")
            
    except requests.RequestException as e:
        print(f"Error al obtener datos de {matrix}-attack: {e}")
        return None

In [9]:
def save_df_as_csv(df, name, aux_folder='stix2', aux_folder_2=''):
    now = datetime.datetime.now()
    if not os.path.exists(os.path.join(os.getcwd(), 'outputs', aux_folder, aux_folder_2)):
        os.makedirs(os.path.join(os.getcwd(), 'outputs', aux_folder,aux_folder_2))
    file_to_save = os.path.join(os.getcwd(), 'outputs', aux_folder, aux_folder_2, (name+ '_'+ now.strftime('%d%m%Y_%H%M')+'h.csv'))
    df.to_csv(file_to_save, sep=';', encoding='utf8', index=False, quoting=csv.QUOTE_NONNUMERIC)
    print('Archivo guardado correctamente '+ name + '_' +now.strftime('%d%m%Y_%H%M')+'h.csv')

In [10]:
def get_list_techniques_from_stix2(src, include="both"):
    '''
    Función encargada de la obtención de la lista de técnicas de los datos facilitados. Por defecto se retornan tanto técnicas como subtécnicas. Retorna una lista de strings con el id correspondiente a cada técnica.
    '''
    if include == "techniques":
        query_results = src.query([
            Filter('type', '=', 'attack-pattern'),
            Filter('x_mitre_is_subtechnique', '=', False)
        ])
    elif include == "subtechniques":
        query_results = src.query([
            Filter('type', '=', 'attack-pattern'),
            Filter('x_mitre_is_subtechnique', '=', True)
        ])
    elif include == "both":
        query_results = src.query([
            Filter('type', '=', 'attack-pattern')
        ])
    else:
        raise RuntimeError("Unknown option %s!" % include)
    
    if isinstance(query_results, stix2.datastore.memory.MemoryStore):
        query_results = query_results.query()
        stix2_df = pd.DataFrame(query_results)
    elif isinstance(query_results[0], stix2.v20.sdo.AttackPattern):
        stix2_df = pd.DataFrame(query_results)

    stix2_df['technique_id'] = stix2_df['external_references'].apply(lambda refs: refs[0].external_id if refs else None)
    stix2_df = stix2_df[(stix2_df['revoked']!=True)&(stix2_df['x_mitre_deprecated']!=True)]
    stix2_df['technique_id'] = stix2_df['technique_id'].str.upper()
    techniques = sorted(stix2_df['technique_id'].drop_duplicates(), key=len, reverse=True)

    return techniques

In [11]:
def get_CP_ttps_with_rules(mitre_matrix, way='file'):
    '''
    Función encargada de obtner el listado de TTP's para las que desde CP disponemos de reglas de detección. Para la ejecutarla correctamente debe de haberse ejecutado previamente el código del bloque get_rules_and_classify_by_ttp. Hay dos formas de evaluar las técnicas disponibles y son gestionadas por el parámetro "way". Por defecto el modo es "file", lo que nos indica que se va a evaluar uno de los ficheros generados de resumen de reglas por TTP clasificadas. En el caso de que el parámetro "way" recoja el valor "path", lo que hará es identificar las reglas meciante la exploración de subdirectorios.  
    '''
    try:
        main_path = os.path.join(os.path.dirname(os.getcwd()), 'get_rules_and_classify_by_ttp', 'outputs', mitre_matrix)
        
        if not os.path.exists(main_path):
            raise ValueError(f'No existe la path: {main_path}')
        
        if way == 'file':
            file_path = os.path.join(main_path, f'{mitre_matrix}-ttp_all_classified_rules.csv')  # Corrección aquí
            file = pd.read_csv(file_path, sep=';')
            file = file[file['ttp'] != 'T0000']  # Filtramos la técnica ficticia donde metemos las reglas que no han sido mapeadas
            CP_techniques = file['ttp'].unique().tolist()  # Convertimos en lista de items únicos la columna que informa de la ttp
            CP_techniques = sorted(CP_techniques, key=len, reverse=True)  # Ordenamos por longitud de caracteres para que cuando sea utilizada esta lista primero se evalúen las subtécnicas.

        elif way == 'path':
            CP_techniques = get_unique_ttps_generated(get_list_of_files_sub(main_path))
            CP_techniques = sorted(CP_techniques, key=len, reverse=True)
            CP_techniques = [ttp for ttp in CP_techniques if ttp.startswith('T')]
            CP_techniques = [ttp for ttp in CP_techniques if ttp != 'T0000']

        else:
            CP_techniques = []
            
    except ValueError as e:
        print(f'{e}')
        CP_techniques = []
    
    return CP_techniques

In [12]:
def get_CP_and_NOCP_techniques_from_df(cp_techniques_list, df):
    '''
    Función encargada de filtrar un df dado el cual contiene el campo techniques_ID por las técnicas disponibles en CP (estas son facilitadas como parámetro en forma de lista). 
    '''
    cp_df = pd.DataFrame()
    nocp_df = pd.DataFrame() 
    try:
        cp_df = df[df['technique_ID'].isin(cp_techniques_list)]
        nocp_df = df[~df['technique_ID'].isin(cp_techniques_list)]
    except:
        print('No se ha podido ejecutar la operación.')
    return cp_df, nocp_df

In [13]:
def get_techniques_tactics_dataframe(matrix_store, cp_techniques_list):
    '''
    Función que, dada la matriz MITRE (entreprise, ics o mobile) devuelve un dataframe con la relación N:N, 1:N y N:1 entre técnicas y tácticas 
    '''
    # Verificamos que no haya habido algun error en la generación de la matriz
    if matrix_store is None:
        raise ValueError("El MemoryStore está vacío o no se ha inicializado correctamente.")

    # Obtenemos las técnicas de la matriz
    techniques = matrix_store.query([
        Filter('type', '=', 'attack-pattern')
    ])
    # Obtenemos las tácticas de la matriz
    tactics = matrix_store.query([
        Filter('type', '=', 'x-mitre-tactic')
    ])
    
    # Mediante diccionarios mapeamos la información que necesitamos
    technique_dict = {
        tech['external_references'][0]['external_id']: {
            'name': tech['name'],
            'deprecated': tech.get('x_mitre_deprecated', False),
            'revoked': tech.get('revoked', False)
        }
        for tech in techniques
        if 'external_references' in tech and len(tech['external_references']) > 0
    }
    tactic_dict = {
        tac['external_references'][0]['external_id']: tac['name']
        for tac in tactics
        if 'external_references' in tac and len(tac['external_references']) > 0
    }
    # Lista para almacenar las relaciones
    data = []

    # Recorrer todas las técnicas para encontrar sus relaciones con tácticas
    for technique in techniques:
        if 'kill_chain_phases' in technique:
            for phase in technique['kill_chain_phases']:
                # Cada fase representa una relación técnica-táctica
                tactic_shortname = phase['phase_name']
                external_id = technique['external_references'][0]['external_id'] if 'external_references' in technique and len(technique['external_references']) > 0 else None
                
                # Buscar el ID externo de la táctica correspondiente
                tactic_id = next((tac['external_references'][0]['external_id'] for tac in tactics if tac['x_mitre_shortname'] == tactic_shortname), None)
                # Si no hubiera ninguna relación
                if not external_id or not tactic_id:
                    continue
                # Agregamos la relación a la lista
                data.append({
                    "technique_ID": external_id,
                    "technique": technique_dict[external_id]['name'],
                    "tactic_ID": tactic_id,
                    "tactic": tactic_dict[tactic_id],
                    "technique_deprecated": technique_dict[external_id]['deprecated'],
                    "technique_revoked": technique_dict[external_id]['revoked']
                })
    
    # Creamos el dataframe final
    techniques_tactics_df = pd.DataFrame(data)
    #Filtramos técnicas deprecadas
    techniques_tactics_NN_df = techniques_tactics_df[(techniques_tactics_df['technique_deprecated']!=True)&(techniques_tactics_df['technique_revoked']!=True)]

    # Generamos el que finalmente será nuestro df compuesto por tecnicas y tácticas en una relación N:N
    MITRE_techniques_tactics_NN_df = techniques_tactics_NN_df[['technique_ID', 'technique', 'tactic_ID', 'tactic']] # Filtramos las columnas que necesitamos
    # A continuación comenzamos por filtrar la tabla fundamental por las técnicas disponibles y no disponibles en CP
    CP_techniques_tactics_NN_df, NOCP_techniques_tactics_NN_df = get_CP_and_NOCP_techniques_from_df(cp_techniques_list, MITRE_techniques_tactics_NN_df)


    # Para finalizar agregamos por técnica y táctica con el objetivo de poder ofrecer otras tablas valiosas de cara a realizar consultas
    # Generamos el df compuesto por técnica y tácticas en una relación 1:N
    MITRE_technique_tactics_1N_df = MITRE_techniques_tactics_NN_df.groupby('technique_ID', as_index=False).agg({
    'technique':unique_list,
    'tactic_ID': unique_list,
    'tactic': unique_list
    })
    CP_technique_tactics_1N_df = CP_techniques_tactics_NN_df.groupby('technique_ID', as_index=False).agg({
    'technique':unique_list,
    'tactic_ID': unique_list,
    'tactic': unique_list
    })
    NOCP_technique_tactics_1N_df = NOCP_techniques_tactics_NN_df.groupby('technique_ID', as_index=False).agg({
    'technique':unique_list,
    'tactic_ID': unique_list,
    'tactic': unique_list
    })


    # Generamos el df compuesto por tecnicas y tácticas en una relación N:1
    MITRE_techniques_tactic_N1_df = MITRE_techniques_tactics_NN_df.groupby('tactic_ID', as_index=False).agg({
    'tactic':unique_list,
    'technique_ID': unique_list,
    'technique': unique_list
    })
    CP_techniques_tactic_N1_df = CP_techniques_tactics_NN_df.groupby('tactic_ID', as_index=False).agg({
    'tactic':unique_list,
    'technique_ID': unique_list,
    'technique': unique_list
    })
    NOCP_techniques_tactic_N1_df = NOCP_techniques_tactics_NN_df.groupby('tactic_ID', as_index=False).agg({
    'tactic':unique_list,
    'technique_ID': unique_list,
    'technique': unique_list
    })

    print('Dataframes generados correctamente!')

    return MITRE_techniques_tactics_NN_df, CP_techniques_tactics_NN_df, NOCP_techniques_tactics_NN_df, MITRE_technique_tactics_1N_df, CP_technique_tactics_1N_df, NOCP_technique_tactics_1N_df, MITRE_techniques_tactic_N1_df, CP_techniques_tactic_N1_df, NOCP_techniques_tactic_N1_df

In [14]:
def get_techniques_datasources_dataframe(matrix_store, cp_techniques_list):
    '''
    Función que, dada la matriz MITRE (entreprise, ics o mobile) devuelve un dataframe con la relación N:N, 1:N y N:1 entre técnicas y data sources 
    '''
    # Verificamos que no haya habido algun error en la generación de la matriz
    if matrix_store is None:
        raise ValueError("El MemoryStore está vacío o no se ha inicializado correctamente.")

    # Obtenemos las técnicas de la matriz
    techniques = matrix_store.query([
        Filter('type', '=', 'attack-pattern')
    ])
    techniques_data = []
    for technique in techniques:
        ttp_name = ''
        ttp_id = ''
        ttp_ds_name = ''
        ttp_revoked = ''
        ttp_deprecated = ''
        if 'external_references' in technique:
            ttp_id = technique['external_references'][0]['external_id']
        if 'name' in technique:
            ttp_name = technique['name']
        if 'x_mitre_data_sources' in technique: 
            ttp_ds_name = technique['x_mitre_data_sources']
        if 'revoked' in technique: 
            ttp_revoked = technique['revoked']
        if 'x_mitre_deprecated' in technique: 
            ttp_deprecated = technique['x_mitre_deprecated']

        techniques_data.append({
            "technique_ID": ttp_id,
            "technique": ttp_name,
            "data_source": ttp_ds_name,
            "technique_deprecated": ttp_deprecated,
            "technique_revoked": ttp_revoked
        })
    # Generamos df a partir de los datos recopilados
    techniques_df = pd.DataFrame(techniques_data)
    # Transformaciones de la tabla de relaciones de técnicas
    techniques_df = techniques_df.explode('data_source')
    techniques_df['data_source'] = techniques_df['data_source'].str.split(':').str[0] # El datasource al que aplica cada 
    techniques_df = techniques_df[(techniques_df['technique_deprecated']!=True)&(techniques_df['technique_revoked']!=True)]
    techniques_df = techniques_df[['technique_ID', 'technique', 'data_source']]

    # Obtenemos los data sources de la matriz
    data_sources = matrix_store.query([
        Filter('type', '=', 'x-mitre-data-source')
    ])

    ds_data = []

    for data_source in data_sources:
        ds_id = ''
        ds_name = ''
        ds_revoked = ''
        ds_deprecated = ''
        if 'external_references' in data_source:
            ds_id = data_source['external_references'][0]['external_id']
        if 'name' in data_source:
            ds_name = data_source['name']
        if 'revoked' in data_source:
            ds_revoked = data_source['revoked']
        if 'x_mitre_deprecated' in data_source:
            ds_deprecated = data_source['x_mitre_deprecated']

        ds_data.append({
            "data_source_ID": ds_id,
            "data_source": ds_name,
            "data_source_deprecated": ds_deprecated,
            "data_source_revoked": ds_revoked
        })
    # Generamos df a partir de los datos recopilados
    data_sources_df = pd.DataFrame(ds_data)
    # Transformaciones de la tabla de relaciones de data sources
    data_sources_df = data_sources_df[(data_sources_df['data_source_deprecated']!=True)&(data_sources_df['data_source_revoked']!=True)]
    data_sources_df = data_sources_df[['data_source_ID', 'data_source']] 
    data_sources_df = data_sources_df.sort_values(by='data_source_ID')

    # Unimos las tablas
    techniques_data_sources_df = pd.merge(techniques_df, data_sources_df, on='data_source', how='left')
    techniques_data_sources_df = techniques_data_sources_df.drop_duplicates()

    # Generamos el que finalmente será nuestro df compuesto por tecnicas y tácticas en una relación N:N
    MITRE_techniques_datasources_NN_df = techniques_data_sources_df[['technique_ID', 'technique', 'data_source_ID', 'data_source']]
    MITRE_techniques_datasources_NN_df = MITRE_techniques_datasources_NN_df.sort_values(by='technique_ID').reset_index(drop=True)
    MITRE_techniques_datasources_NN_df

    # A continuación comenzamos por filtrar la tabla fundamental por las técnicas disponibles y no disponibles en CP
    CP_techniques_datasources_NN_df, NOCP_techniques_datasources_NN_df = get_CP_and_NOCP_techniques_from_df(cp_techniques_list, MITRE_techniques_datasources_NN_df)


    # Para finalizar agregamos por técnica y data sources con el objetivo de poder ofrecer otras tablas valiosas de cara a realizar consultas
    # Generamos el df compuesto por técnica y data sources en una relación 1:N
    MITRE_technique_datasources_1N_df = MITRE_techniques_datasources_NN_df.groupby('technique_ID', as_index=False).agg({
    'technique':unique_list,
    'data_source_ID': unique_list,
    'data_source': unique_list
    })
    CP_technique_datasources_1N_df = CP_techniques_datasources_NN_df.groupby('technique_ID', as_index=False).agg({
    'technique':unique_list,
    'data_source_ID': unique_list,
    'data_source': unique_list
    })
    NOCP_technique_datasources_1N_df = NOCP_techniques_datasources_NN_df.groupby('technique_ID', as_index=False).agg({
    'technique':unique_list,
    'data_source_ID': unique_list,
    'data_source': unique_list
    })

    # Generamos el df compuesto por tecnicas y data sources en una relación N:1
    MITRE_techniques_tactic_N1_df = MITRE_techniques_datasources_NN_df.groupby('data_source_ID', as_index=False).agg({
    'data_source':unique_list,
    'technique_ID': unique_list,
    'technique': unique_list
    })
    CP_techniques_tactic_N1_df = CP_techniques_datasources_NN_df.groupby('data_source_ID', as_index=False).agg({
    'data_source':unique_list,
    'technique_ID': unique_list,
    'technique': unique_list
    })
    NOCP_techniques_tactic_N1_df = NOCP_techniques_datasources_NN_df.groupby('data_source_ID', as_index=False).agg({
    'data_source':unique_list,
    'technique_ID': unique_list,
    'technique': unique_list
    })

    print('Dataframes generados correctamente!')

    return MITRE_techniques_datasources_NN_df, CP_techniques_datasources_NN_df, NOCP_techniques_datasources_NN_df, MITRE_technique_datasources_1N_df, CP_technique_datasources_1N_df, NOCP_technique_datasources_1N_df, MITRE_techniques_tactic_N1_df, CP_techniques_tactic_N1_df, NOCP_techniques_tactic_N1_df

In [15]:
def get_techniques_platforms_dataframe(matrix_store, cp_techniques_list):
    '''
    Función que, dada la matriz MITRE (entreprise, ics o mobile) devuelve un dataframe con la relación N:N, 1:N y N:1 entre técnicas y plataformas
    '''
    # Verificamos que no haya habido algun error en la generación de la matriz
    if matrix_store is None:
        raise ValueError("El MemoryStore está vacío o no se ha inicializado correctamente.")

    # Obtenemos las técnicas de la matriz
    techniques = matrix_store.query([
        Filter('type', '=', 'attack-pattern')
    ])
    techniques_data = []
    for technique in techniques:
        ttp_name = ''
        ttp_id = ''
        ttp_platform_name = ''
        ttp_revoked = ''
        ttp_deprecated = ''
        if 'external_references' in technique:
            ttp_id = technique['external_references'][0]['external_id']
        if 'name' in technique:
            ttp_name = technique['name']
        if 'x_mitre_data_sources' in technique: 
            ttp_platform_name = technique['x_mitre_platforms']
        if 'revoked' in technique: 
            ttp_revoked = technique['revoked']
        if 'x_mitre_deprecated' in technique: 
            ttp_deprecated = technique['x_mitre_deprecated']

        techniques_data.append({
            "technique_ID": ttp_id,
            "technique": ttp_name,
            "platform": ttp_platform_name,
            "technique_deprecated": ttp_deprecated,
            "technique_revoked": ttp_revoked
        })
    # Generamos df a partir de los datos recopilados
    techniques_df = pd.DataFrame(techniques_data)

    techniques_df = techniques_df[(techniques_df['technique_deprecated']!=True)&(techniques_df['technique_revoked']!=True)]
    techniques_df = techniques_df[['technique_ID', 'technique', 'platform']]
    techniques_df = techniques_df.explode('platform')
    MITRE_techniques_platforms_NN_df = techniques_df.drop_duplicates()
    MITRE_techniques_platforms_NN_df = MITRE_techniques_platforms_NN_df.sort_values(by='technique_ID').reset_index(drop=True)

    # A continuación comenzamos por filtrar la tabla fundamental por las técnicas disponibles y no disponibles en CP
    CP_techniques_platforms_NN_df, NOCP_techniques_platforms_NN_df = get_CP_and_NOCP_techniques_from_df(cp_techniques_list, MITRE_techniques_platforms_NN_df)


    # Para finalizar agregamos por técnica y plataformas con el objetivo de poder ofrecer otras tablas valiosas de cara a realizar consultas
    # Generamos el df compuesto por técnica y plataformas en una relación 1:N
    MITRE_technique_platforms_1N_df = MITRE_techniques_platforms_NN_df.groupby('technique_ID', as_index=False).agg({
    'technique':unique_list,
    'platform': unique_list
    })
    CP_technique_platforms_1N_df = CP_techniques_platforms_NN_df.groupby('technique_ID', as_index=False).agg({
    'technique':unique_list,
    'platform': unique_list
    })
    NOCP_technique_platforms_1N_df = NOCP_techniques_platforms_NN_df.groupby('technique_ID', as_index=False).agg({
    'technique':unique_list,
    'platform': unique_list
    })

    # Generamos el df compuesto por tecnicas y plataformas en una relación N:1
    MITRE_techniques_platform_N1_df = MITRE_techniques_platforms_NN_df.groupby('platform', as_index=False).agg({
    'technique_ID': unique_list,
    'technique': unique_list
    })
    CP_techniques_platform_N1_df = CP_techniques_platforms_NN_df.groupby('platform', as_index=False).agg({
    'technique_ID': unique_list,
    'technique': unique_list
    })
    NOCP_techniques_platform_N1_df = NOCP_techniques_platforms_NN_df.groupby('platform', as_index=False).agg({
    'technique_ID': unique_list,
    'technique': unique_list
    })

    print('Dataframes generados correctamente!')

    return MITRE_techniques_platforms_NN_df, CP_techniques_platforms_NN_df, NOCP_techniques_platforms_NN_df, MITRE_technique_platforms_1N_df, CP_technique_platforms_1N_df, NOCP_technique_platforms_1N_df, MITRE_techniques_platform_N1_df, CP_techniques_platform_N1_df, NOCP_techniques_platform_N1_df

#### **Funciones en desarrollo**

### **Parámetros**

In [16]:
matrix = 'enterprise'
save_as_csv = True
debug_df = True # Parámetro para controlar el printeado de df

### **Ejecución principal**

#### **1. Elementos generales**

##### **1.1 Generación de la matriz MITRE**

In [17]:
mitre_matrix = get_data_from_branch(matrix)
mitre_matrix

##### **1.2 Generación de la lista de técnicas (ID) de la matriz MITRE seleccionada**

In [18]:
techniques_list = get_list_techniques_from_stix2(mitre_matrix,'both')
print(f"Se han generado la lista de técnicas (ID) para la matriz {matrix.upper()} que contiene un total de {len(techniques_list)} TTP's")

Se han generado la lista de técnicas (ID) para la matriz ENTERPRISE que contiene un total de 637 TTP's


##### **1.3 Obtención de las TTP disponbles en Cyber Proof con regla de detección**

In [19]:
cp_techniques =  get_CP_ttps_with_rules(matrix, way='file')
print(f"Se han obtenido un total de {len(cp_techniques)} TTP's con regla de detección asociada disponibles en CP.")

Se han obtenido un total de 521 TTP's con regla de detección asociada disponibles en CP.


### **2. Relación técnicas - tácticas**

#### **Generación de las tablas**

In [20]:
MITRE_techniques_tactics_NN_df, CP_techniques_tactics_NN_df, NOCP_techniques_tactics_NN_df, MITRE_technique_tactics_1N_df, CP_technique_tactics_1N_df, NOCP_technique_tactics_1N_df, MITRE_techniques_tactic_N1_df, CP_techniques_tactic_N1_df, NOCP_techniques_tactic_N1_df = get_techniques_tactics_dataframe(mitre_matrix, cp_techniques)

Dataframes generados correctamente!


#### **Debug**

In [21]:
if debug_df:
    display(MITRE_techniques_tactics_NN_df.head(3))
    display(CP_techniques_tactics_NN_df.head(3))
    display(NOCP_techniques_tactics_NN_df.head(3))
    print(f'{MITRE_techniques_tactics_NN_df.shape}, {CP_techniques_tactics_NN_df.shape}, {NOCP_techniques_tactics_NN_df.shape}')

,technique_ID,technique,tactic_ID,tactic
0,T1055.011,Extra Window Memory Injection,TA0005,Defense Evasion
1,T1055.011,Extra Window Memory Injection,TA0004,Privilege Escalation
2,T1053.005,Scheduled Task,TA0002,Execution


,technique_ID,technique,tactic_ID,tactic
0,T1055.011,Extra Window Memory Injection,TA0005,Defense Evasion
1,T1055.011,Extra Window Memory Injection,TA0004,Privilege Escalation
2,T1053.005,Scheduled Task,TA0002,Execution


,technique_ID,technique,tactic_ID,tactic
5,T1205.002,Socket Filters,TA0005,Defense Evasion
6,T1205.002,Socket Filters,TA0003,Persistence
7,T1205.002,Socket Filters,TA0011,Command and Control


(830, 4), (686, 4), (144, 4)


In [22]:
if debug_df:
    display(MITRE_technique_tactics_1N_df.head(3))
    display(CP_technique_tactics_1N_df.head(3))
    display(NOCP_technique_tactics_1N_df.head(3))
    print(f'{MITRE_technique_tactics_1N_df.shape}, {CP_technique_tactics_1N_df.shape}, {NOCP_technique_tactics_1N_df.shape}')

,technique_ID,technique,tactic_ID,tactic
0,T1001,[Data Obfuscation],[TA0011],[Command and Control]
1,T1001.001,[Junk Data],[TA0011],[Command and Control]
2,T1001.002,[Steganography],[TA0011],[Command and Control]


,technique_ID,technique,tactic_ID,tactic
0,T1001,[Data Obfuscation],[TA0011],[Command and Control]
1,T1001.002,[Steganography],[TA0011],[Command and Control]
2,T1001.003,[Protocol Impersonation],[TA0011],[Command and Control]


,technique_ID,technique,tactic_ID,tactic
0,T1001.001,[Junk Data],[TA0011],[Command and Control]
1,T1011.001,[Exfiltration Over Bluetooth],[TA0010],[Exfiltration]
2,T1016.002,[Wi-Fi Discovery],[TA0007],[Discovery]


(637, 4), (521, 4), (116, 4)


In [23]:
if debug_df:
    display(MITRE_techniques_tactic_N1_df.head(3))
    display(CP_techniques_tactic_N1_df.head(3))
    display(NOCP_techniques_tactic_N1_df.head(3))
    print(f'{MITRE_techniques_tactic_N1_df.shape}, {CP_techniques_tactic_N1_df.shape}, {NOCP_techniques_tactic_N1_df.shape}')

,tactic_ID,tactic,technique_ID,technique
0,TA0001,[Initial Access],"[T1566.004, T1659, T1091, T1566, T1199, T1078....","[Hardware Additions, Phishing, Default Account..."
1,TA0002,[Execution],"[T1059.002, T1059, T1609, T1059.004, T1059.010...","[Shared Modules, PowerShell, User Execution, C..."
2,TA0003,[Persistence],"[T1205.002, T1546.007, T1547.005, T1546, T1547...","[Password Filter DLL, Outlook Home Page, Bootk..."


,tactic_ID,tactic,technique_ID,technique
0,TA0001,[Initial Access],"[T1566.002, T1566.003, T1091, T1189, T1195.001...","[External Remote Services, Compromise Software..."
1,TA0002,[Execution],"[T1059.002, T1059, T1609, T1059.004, T1559.002...","[Shared Modules, PowerShell, User Execution, C..."
2,TA0003,[Persistence],"[T1546.007, T1547.005, T1546, T1547.006, T1546...","[Password Filter DLL, Bootkit, Image File Exec..."


,tactic_ID,tactic,technique_ID,technique
0,TA0001,[Initial Access],"[T1566.004, T1195.003, T1659]","[Spearphishing Voice, Content Injection, Compr..."
1,TA0002,[Execution],"[T1559.003, T1059.008, T1059.010, T1651]","[XPC Services, Cloud Administration Command, A..."
2,TA0003,[Persistence],"[T1205.002, T1098.006, T1653, T1037.002, T1556...","[Additional Container Cluster Roles, Outlook H..."


(14, 4), (14, 4), (13, 4)


#### **Guardado**

In [24]:
if save_as_csv:
    save_df_as_csv(MITRE_techniques_tactics_NN_df, f'[MITRE]_{matrix}_techniques_tactics_NN','stix2', 'Techniques_Tactics')
    save_df_as_csv(CP_techniques_tactics_NN_df, f'[CP]_{matrix}_techniques_tactics_NN','stix2', 'Techniques_Tactics')
    save_df_as_csv(NOCP_techniques_tactics_NN_df, f'[NOCP]_{matrix}_techniques_tactics_NN','stix2', 'Techniques_Tactics')

    save_df_as_csv(MITRE_technique_tactics_1N_df, f'[MITRE]_{matrix}_technique_tactics_1N','stix2', 'Techniques_Tactics')
    save_df_as_csv(CP_technique_tactics_1N_df, f'[CP]_{matrix}_technique_tactics_1N','stix2', 'Techniques_Tactics')
    save_df_as_csv(NOCP_technique_tactics_1N_df, f'[NOCP]_{matrix}_technique_tactics_1N','stix2', 'Techniques_Tactics')

    save_df_as_csv(MITRE_techniques_tactic_N1_df, f'[MITRE]_{matrix}_techniques_tactic_N1','stix2', 'Techniques_Tactics')
    save_df_as_csv(CP_techniques_tactic_N1_df, f'[CP]_{matrix}_techniques_tactic_N1','stix2', 'Techniques_Tactics')
    save_df_as_csv(NOCP_techniques_tactic_N1_df, f'[NOCP]_{matrix}_techniques_tactic_N1','stix2', 'Techniques_Tactics')

Archivo guardado correctamente [MITRE]_enterprise_techniques_tactics_NN_05072024_1205h.csv
Archivo guardado correctamente [CP]_enterprise_techniques_tactics_NN_05072024_1205h.csv
Archivo guardado correctamente [NOCP]_enterprise_techniques_tactics_NN_05072024_1205h.csv
Archivo guardado correctamente [MITRE]_enterprise_technique_tactics_1N_05072024_1205h.csv
Archivo guardado correctamente [CP]_enterprise_technique_tactics_1N_05072024_1205h.csv
Archivo guardado correctamente [NOCP]_enterprise_technique_tactics_1N_05072024_1205h.csv
Archivo guardado correctamente [MITRE]_enterprise_techniques_tactic_N1_05072024_1205h.csv
Archivo guardado correctamente [CP]_enterprise_techniques_tactic_N1_05072024_1205h.csv
Archivo guardado correctamente [NOCP]_enterprise_techniques_tactic_N1_05072024_1205h.csv


### **3. Relación técnicas - data sources**

#### **Generación de las tablas**

In [25]:
MITRE_techniques_datasources_NN_df, CP_techniques_datasources_NN_df, NOCP_techniques_datasources_NN_df, MITRE_technique_datasources_1N_df, CP_technique_datasources_1N_df, NOCP_technique_datasources_1N_df, MITRE_techniques_datasource_N1_df, CP_techniques_datasource_N1_df, NOCP_techniques_datasource_N1_df = get_techniques_datasources_dataframe(mitre_matrix, cp_techniques)

Dataframes generados correctamente!


#### **Debug**

In [26]:
if debug_df:
    display(MITRE_techniques_datasources_NN_df.head(3))
    display(CP_techniques_datasources_NN_df.head(3))
    display(NOCP_techniques_datasources_NN_df.head(3))
    print(f'{MITRE_techniques_datasources_NN_df.shape}, {CP_techniques_datasources_NN_df.shape}, {NOCP_techniques_datasources_NN_df.shape}')

,technique_ID,technique,data_source_ID,data_source
0,T1001,Data Obfuscation,DS0029,Network Traffic
1,T1001.001,Junk Data,DS0029,Network Traffic
2,T1001.002,Steganography,DS0029,Network Traffic


,technique_ID,technique,data_source_ID,data_source
0,T1001,Data Obfuscation,DS0029,Network Traffic
2,T1001.002,Steganography,DS0029,Network Traffic
3,T1001.003,Protocol Impersonation,DS0029,Network Traffic


,technique_ID,technique,data_source_ID,data_source
1,T1001.001,Junk Data,DS0029,Network Traffic
44,T1011.001,Exfiltration Over Bluetooth,DS0029,Network Traffic
45,T1011.001,Exfiltration Over Bluetooth,DS0017,Command


(1594, 4), (1401, 4), (193, 4)


In [27]:
if debug_df:
    display(MITRE_technique_datasources_1N_df.head(3))
    display(CP_technique_datasources_1N_df.head(3))
    display(NOCP_technique_datasources_1N_df.head(3))
    print(f'{MITRE_technique_datasources_1N_df.shape}, {CP_technique_datasources_1N_df.shape}, {NOCP_technique_datasources_1N_df.shape}')

,technique_ID,technique,data_source_ID,data_source
0,T1001,[Data Obfuscation],[DS0029],[Network Traffic]
1,T1001.001,[Junk Data],[DS0029],[Network Traffic]
2,T1001.002,[Steganography],[DS0029],[Network Traffic]


,technique_ID,technique,data_source_ID,data_source
0,T1001,[Data Obfuscation],[DS0029],[Network Traffic]
1,T1001.002,[Steganography],[DS0029],[Network Traffic]
2,T1001.003,[Protocol Impersonation],[DS0029],[Network Traffic]


,technique_ID,technique,data_source_ID,data_source
0,T1001.001,[Junk Data],[DS0029],[Network Traffic]
1,T1011.001,[Exfiltration Over Bluetooth],"[DS0022, DS0017, DS0029]","[Network Traffic, Command, File]"
2,T1016.002,[Wi-Fi Discovery],"[DS0009, DS0017]","[Process, Command]"


(637, 4), (521, 4), (116, 4)


In [28]:
if debug_df:
    display(MITRE_techniques_datasource_N1_df.head(3))
    display(CP_techniques_datasource_N1_df.head(3))
    display(NOCP_techniques_datasource_N1_df.head(3))
    print(f'{MITRE_techniques_datasource_N1_df.shape}, {CP_techniques_datasource_N1_df.shape}, {NOCP_techniques_datasource_N1_df.shape}')

,data_source_ID,data_source,technique_ID,technique
0,DS0001,[Firmware],"[T1542.005, T1014, T1542.001, T1542, T1542.002...","[TFTP Boot, ROMMONkit, Firmware Corruption, Sy..."
1,DS0002,[User Account],"[T1550.002, T1098.006, T1207, T1531, T1110, T1...","[Pass the Hash, SAML Tokens, Default Accounts,..."
2,DS0003,[Scheduled Job],"[T1053, T1053.002, T1070.009, T1053.003, T1053...","[Scheduled Task/Job, Clear Persistence, Masque..."


,data_source_ID,data_source,technique_ID,technique
0,DS0001,[Firmware],"[T1542.005, T1014, T1542.001, T1542, T1564.005...","[TFTP Boot, Firmware Corruption, System Firmwa..."
1,DS0002,[User Account],"[T1550.002, T1207, T1531, T1110, T1538, T1136,...","[Pass the Hash, SAML Tokens, Default Accounts,..."
2,DS0003,[Scheduled Job],"[T1053, T1053.002, T1070.009, T1053.003, T1053...","[Scheduled Task/Job, Clear Persistence, Masque..."


,data_source_ID,data_source,technique_ID,technique
0,DS0001,[Firmware],"[T1542.002, T1542.004]","[Component Firmware, ROMMONkit]"
1,DS0002,[User Account],"[T1556.005, T1098.006, T1548.005, T1098.005]","[Additional Container Cluster Roles, Device Re..."
2,DS0004,[Malware Repository],[T1587.002],[Code Signing Certificates]


(37, 4), (37, 4), (23, 4)


#### **Guardado**

In [29]:
if save_as_csv:
    save_df_as_csv(MITRE_techniques_datasources_NN_df, f'[MITRE]_{matrix}_techniques_datasources_NN','stix2', 'Techniques_DataSources')
    save_df_as_csv(CP_techniques_datasources_NN_df, f'[CP]_{matrix}_techniques_datasources_NN','stix2', 'Techniques_DataSources')
    save_df_as_csv(NOCP_techniques_datasources_NN_df, f'[NOCP]_{matrix}_techniques_datasources_NN','stix2', 'Techniques_DataSources')

    save_df_as_csv(MITRE_technique_datasources_1N_df, f'[MITRE]_{matrix}_technique_datasources_1N','stix2', 'Techniques_DataSources')
    save_df_as_csv(CP_technique_datasources_1N_df, f'[CP]_{matrix}_technique_datasources_1N','stix2', 'Techniques_DataSources')
    save_df_as_csv(NOCP_technique_datasources_1N_df, f'[NOCP]_{matrix}_technique_datasources_1N','stix2', 'Techniques_DataSources')

    save_df_as_csv(MITRE_techniques_datasource_N1_df, f'[MITRE]_{matrix}_techniques_datasource_N1','stix2', 'Techniques_DataSources')
    save_df_as_csv(CP_techniques_datasource_N1_df, f'[CP]_{matrix}_techniques_datasource_N1','stix2', 'Techniques_DataSources')
    save_df_as_csv(NOCP_techniques_datasource_N1_df, f'[NOCP]_{matrix}_techniques_datasource_N1','stix2', 'Techniques_DataSources')

Archivo guardado correctamente [MITRE]_enterprise_techniques_datasources_NN_05072024_1205h.csv
Archivo guardado correctamente [CP]_enterprise_techniques_datasources_NN_05072024_1205h.csv
Archivo guardado correctamente [NOCP]_enterprise_techniques_datasources_NN_05072024_1205h.csv
Archivo guardado correctamente [MITRE]_enterprise_technique_datasources_1N_05072024_1205h.csv
Archivo guardado correctamente [CP]_enterprise_technique_datasources_1N_05072024_1205h.csv
Archivo guardado correctamente [NOCP]_enterprise_technique_datasources_1N_05072024_1205h.csv
Archivo guardado correctamente [MITRE]_enterprise_techniques_datasource_N1_05072024_1205h.csv
Archivo guardado correctamente [CP]_enterprise_techniques_datasource_N1_05072024_1205h.csv
Archivo guardado correctamente [NOCP]_enterprise_techniques_datasource_N1_05072024_1205h.csv


### **4. Relación técnicas - plataformas**

#### **Generación de las tablas**

In [30]:
MITRE_techniques_platforms_NN_df, CP_techniques_platforms_NN_df, NOCP_techniques_platforms_NN_df, MITRE_technique_platforms_1N_df, CP_technique_platforms_1N_df, NOCP_technique_platforms_1N_df, MITRE_techniques_platform_N1_df, CP_techniques_platform_N1_df, NOCP_techniques_platform_N1_df = get_techniques_platforms_dataframe(mitre_matrix, cp_techniques)

Dataframes generados correctamente!


#### **Debug**

In [31]:
if debug_df:
    display(MITRE_techniques_platforms_NN_df.head(3))
    display(CP_techniques_platforms_NN_df.head(3))
    display(NOCP_techniques_platforms_NN_df.head(3))
    print(f'{MITRE_techniques_platforms_NN_df.shape}, {CP_techniques_platforms_NN_df.shape}, {NOCP_techniques_platforms_NN_df.shape}')

,technique_ID,technique,platform
0,T1001,Data Obfuscation,Linux
1,T1001,Data Obfuscation,macOS
2,T1001,Data Obfuscation,Windows


,technique_ID,technique,platform
0,T1001,Data Obfuscation,Linux
1,T1001,Data Obfuscation,macOS
2,T1001,Data Obfuscation,Windows


,technique_ID,technique,platform
3,T1001.001,Junk Data,Windows
4,T1001.001,Junk Data,Linux
5,T1001.001,Junk Data,macOS


(1654, 3), (1436, 3), (218, 3)


In [32]:
if debug_df:
    display(MITRE_technique_platforms_1N_df.head(3))
    display(CP_technique_platforms_1N_df.head(3))
    display(NOCP_technique_platforms_1N_df.head(3))
    print(f'{MITRE_technique_platforms_1N_df.shape}, {CP_technique_platforms_1N_df.shape}, {NOCP_technique_platforms_1N_df.shape}')

,technique_ID,technique,platform
0,T1001,[Data Obfuscation],"[macOS, Windows, Linux]"
1,T1001.001,[Junk Data],"[macOS, Windows, Linux]"
2,T1001.002,[Steganography],"[macOS, Windows, Linux]"


,technique_ID,technique,platform
0,T1001,[Data Obfuscation],"[macOS, Windows, Linux]"
1,T1001.002,[Steganography],"[macOS, Windows, Linux]"
2,T1001.003,[Protocol Impersonation],"[macOS, Windows, Linux]"


,technique_ID,technique,platform
0,T1001.001,[Junk Data],"[macOS, Windows, Linux]"
1,T1011.001,[Exfiltration Over Bluetooth],"[macOS, Windows, Linux]"
2,T1016.002,[Wi-Fi Discovery],"[macOS, Windows, Linux]"


(637, 3), (521, 3), (116, 3)


In [33]:
if debug_df:
    display(MITRE_techniques_platform_N1_df.head(3))
    display(CP_techniques_platform_N1_df.head(3))
    display(NOCP_techniques_platform_N1_df.head(3))
    print(f'{MITRE_techniques_platform_N1_df.shape}, {CP_techniques_platform_N1_df.shape}, {NOCP_techniques_platform_N1_df.shape}')

,platform,technique_ID,technique
0,,"[T1590.005, T1586.003, T1590, T1597, T1588.006...","[Scan Databases, DNS/Passive DNS, Exploits, Ga..."
1,Azure AD,"[T1059, T1550.001, T1498, T1110, T1498.001, T1...","[Command and Scripting Interpreter, SAML Token..."
2,Containers,"[T1612, T1098.006, T1543, T1550.001, T1485, T1...","[Inhibit System Recovery, User Execution, Masq..."


,platform,technique_ID,technique
0,,"[T1590.005, T1586.003, T1590, T1597, T1590.003...","[Gather Victim Network Information, Network Se..."
1,Azure AD,"[T1059, T1550.001, T1498, T1110, T1498.001, T1...","[Command and Scripting Interpreter, SAML Token..."
2,Containers,"[T1612, T1543, T1550.001, T1485, T1498, T1609,...","[Inhibit System Recovery, User Execution, Masq..."


,platform,technique_ID,technique
0,,"[T1596.001, T1585.003, T1596.002, T1596.004, T...","[Scan Databases, DNS Server, Social Media, WHO..."
1,Azure AD,"[T1548.005, T1098.005, T1556.009]","[Device Registration, Conditional Access Polic..."
2,Containers,"[T1098.006, T1543.005]","[Additional Container Cluster Roles, Container..."


(12, 3), (12, 3), (12, 3)


#### **Guardado**

In [34]:
if save_as_csv:
    save_df_as_csv(MITRE_techniques_platforms_NN_df, f'[MITRE]_{matrix}_techniques_platforms_NN','stix2', 'Techniques_Platforms')
    save_df_as_csv(CP_techniques_platforms_NN_df, f'[CP]_{matrix}_techniques_platforms_NN','stix2', 'Techniques_Platforms')
    save_df_as_csv(NOCP_techniques_platforms_NN_df, f'[NOCP]_{matrix}_techniques_platforms_NN','stix2', 'Techniques_Platforms')

    save_df_as_csv(MITRE_technique_platforms_1N_df, f'[MITRE]_{matrix}_technique_platforms_1N','stix2', 'Techniques_Platforms')
    save_df_as_csv(CP_technique_platforms_1N_df, f'[CP]_{matrix}_technique_platforms_1N','stix2', 'Techniques_Platforms')
    save_df_as_csv(NOCP_technique_platforms_1N_df, f'[NOCP]_{matrix}_technique_platforms_1N','stix2', 'Techniques_Platforms')

    save_df_as_csv(MITRE_techniques_platform_N1_df, f'[MITRE]_{matrix}_techniques_platform_N1','stix2', 'Techniques_Platforms')
    save_df_as_csv(CP_techniques_platform_N1_df, f'[CP]_{matrix}_techniques_platform_N1','stix2', 'Techniques_Platforms')
    save_df_as_csv(NOCP_techniques_platform_N1_df, f'[NOCP]_{matrix}_techniques_platform_N1','stix2', 'Techniques_Platforms')

Archivo guardado correctamente [MITRE]_enterprise_techniques_platforms_NN_05072024_1205h.csv
Archivo guardado correctamente [CP]_enterprise_techniques_platforms_NN_05072024_1205h.csv
Archivo guardado correctamente [NOCP]_enterprise_techniques_platforms_NN_05072024_1205h.csv
Archivo guardado correctamente [MITRE]_enterprise_technique_platforms_1N_05072024_1205h.csv
Archivo guardado correctamente [CP]_enterprise_technique_platforms_1N_05072024_1205h.csv
Archivo guardado correctamente [NOCP]_enterprise_technique_platforms_1N_05072024_1205h.csv
Archivo guardado correctamente [MITRE]_enterprise_techniques_platform_N1_05072024_1205h.csv
Archivo guardado correctamente [CP]_enterprise_techniques_platform_N1_05072024_1205h.csv
Archivo guardado correctamente [NOCP]_enterprise_techniques_platform_N1_05072024_1205h.csv


In [ ]:
result_query = mitre_matrix.query([Filter('external_references.external_id', '=', 'T1055.011')])
result_query[0]['x_mitre_deprecated']

KeyError: 'x_mitre_deprecated'

In [45]:
# T1542_004 = mitre_matrix.query([Filter("external_references.external_id", "=", "T1542.004")])[0]
T1542_004['revoked']

False

In [47]:
techniques_df.to_csv(r'C:\Users\jelopez\Documents\CyberProof\python\develop\mitre_relationships\outputs\stix2\test_nofilter.csv', sep=';', encoding='utf8', index=False, quoting=csv.QUOTE_NONNUMERIC)

In [ ]:
matrix_store = mitre_matrix
# Verificamos que no haya habido algun error en la generación de la matriz
if matrix_store is None:
    raise ValueError("El MemoryStore está vacío o no se ha inicializado correctamente.")

# Obtenemos las técnicas de la matriz
techniques = matrix_store.query([
    Filter('type', '=', 'attack-pattern')
])
techniques_data = []
ttp_name = ''
ttp_id = ''
ttp_ds_name = ''
ttp_revoked = ''
ttp_deprecated = ''

for technique in techniques:
    if 'external_references' in technique:
        ttp_id = technique['external_references'][0]['external_id']
    if 'name' in technique:
        ttp_name = technique['name']
    if 'x_mitre_data_sources' in technique: 
        ttp_ds_name = technique['x_mitre_data_sources']
    if 'revoked' in technique: 
        ttp_revoked = technique['revoked']
    if 'x_mitre_deprecated' in technique: 
        ttp_deprecated = technique['x_mitre_deprecated']

    techniques_data.append({
        "technique_ID": ttp_id,
        "technique": ttp_name,
        "data_source": ttp_ds_name,
        "technique_deprecated": ttp_deprecated,
        "technique_revoked": ttp_revoked
    })
# Generamos df a partir de los datos recopilados
techniques_df = pd.DataFrame(techniques_data)
# Transformaciones de la tabla de relaciones de técnicas
techniques_df = techniques_df.explode('data_source')
techniques_df['data_source'] = techniques_df['data_source'].str.split(':').str[0] # El datasource al que aplica cada 
techniques_df = techniques_df[(techniques_df['technique_deprecated']!=True)&(techniques_df['technique_revoked']!=True)]
techniques_df = techniques_df[['technique_ID', 'technique', 'data_source']]

# Obtenemos los data sources de la matriz
data_sources = matrix_store.query([
    Filter('type', '=', 'x-mitre-data-source')
])

ds_data = []
ds_id = ''
ds_name = ''
ds_revoked = ''
ds_deprecated = ''
for data_source in data_sources:
    if 'external_references' in data_source:
        ds_id = data_source['external_references'][0]['external_id']
    if 'name' in data_source:
        ds_name = data_source['name']
    if 'revoked' in data_source:
        ds_revoked = data_source['revoked']
    if 'x_mitre_deprecated' in data_source:
        ds_deprecated = data_source['x_mitre_deprecated']

    ds_data.append({
        "data_source_ID": ds_id,
        "data_source": ds_name,
        "data_source_deprecated": ds_deprecated,
        "data_source_revoked": ds_revoked
    })